# SLM — Kaggle 2× T4 DDP Training (150M)

**Before running:** Go to `Settings → Accelerator → GPU T4 × 2`.

This notebook trains a **~150M parameter** SLM using **DistributedDataParallel (DDP)** across
both T4 GPUs via `torchrun`. Each GPU holds a full model copy and synchronises gradients via NCCL.

---

### Session persistence — how it works

Kaggle sessions reset after 12h — all `/kaggle/working/` files are lost unless you save them.
This notebook handles that in two ways:

| Event | What happens |
|---|---|
| **Cell stop button / kernel interrupt** | `train_kaggle.py` catches `SIGTERM` and saves an emergency checkpoint before exiting |
| **8h40m auto-save** | The training script exits cleanly, then cell 8b copies checkpoints to your Kaggle Dataset output (`/kaggle/working/`) which **persists across sessions** |
| **Next session** | Cell 3 restores the checkpoint from your saved dataset; training resumes automatically |

**One-time setup** (first session only):
1. Create a **Kaggle Dataset** called `slm-checkpoints` in your account (can be empty, private is fine).
2. Add it as a **Data Source** to this notebook (right panel → `+ Add Data` → search `slm-checkpoints`).
3. Fill in `GITHUB_REPO` and `KAGGLE_DATASET_DIR` below.

### DDP at a glance
| | Value |
|---|---|
| GPUs | 2× NVIDIA T4 (16 GB each) |
| Backend | NCCL |
| Per-GPU batch | 8 sequences × 8 accum steps |
| Global eff. batch | **131,072 tokens / optimizer step** |
| Model size | **~152M parameters** |
| VRAM per GPU | ~2.1 GB (14 GB headroom) |
| Auto-save at | **8h 40min** (configurable via `SESSION_HOURS`) |

In [ ]:
# ── 0. Configuration — edit these before running ──────────────────────────────

GITHUB_REPO   = 'https://github.com/Aditya7129p/slm.git'  # ← EDIT: your repo URL
                                                          # private: 'https://<TOKEN>@github.com/...'

PROJECT_DIR   = '/kaggle/working/slm'                    # where the repo lives on this session

# Kaggle Dataset used as persistent checkpoint storage across sessions.
# Create a dataset called 'slm-checkpoints' in your account (can be empty).
# Then add it as a data source to this notebook.
# The path below is where Kaggle mounts it read-only:
KAGGLE_DATASET_DIR = '/kaggle/input/slm-checkpoints'     # ← read-only input mount
PERSIST_DIR        = '/kaggle/working/slm_persist'       # ← writable working copy

# Session wall-clock limit before auto-save + exit (hours).
# Default 8h40m leaves ~3h20m to copy checkpoints before the 12h Kaggle limit.
SESSION_HOURS = 8.667

print(f'GITHUB_REPO    : {GITHUB_REPO}')
print(f'PROJECT_DIR    : {PROJECT_DIR}')
print(f'PERSIST_DIR    : {PERSIST_DIR}')
print(f'SESSION_HOURS  : {SESSION_HOURS}h')

In [ ]:
# ── 1. Verify GPU setup ────────────────────────────────────────────────────────
import subprocess, sys, torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('⚠  No GPU detected! Go to Settings → Accelerator → GPU T4 × 2')
    sys.exit(1)

gpus = [line.strip() for line in result.stdout.strip().splitlines()]
print(f'Found {len(gpus)} GPU(s):')
for g in gpus:
    print(f'  {g}')

if len(gpus) < 2:
    print('⚠  Only 1 GPU found — DDP requires 2. Change accelerator to "GPU T4 x2".')
    sys.exit(1)

print(f'\nPyTorch {torch.__version__}  |  CUDA {torch.version.cuda}')
print(f'bfloat16 support : {torch.cuda.is_bf16_supported()}')
print(f'NCCL available   : {torch.distributed.is_nccl_available()}')
print('✔  GPU setup OK')

In [ ]:
# ── 2. Clone / update the SLM repo ────────────────────────────────────────────
import os

if os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    print('Repo already cloned — pulling latest...')
    result = subprocess.run(
        ['git', '-C', PROJECT_DIR, 'pull'],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print('Cloning repo...')
    result = subprocess.run(
        ['git', 'clone', GITHUB_REPO, PROJECT_DIR],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        print('⚠  Clone failed. Check GITHUB_REPO URL and token.')
        sys.exit(1)

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── 3. Restore checkpoints from persistent storage (next-session resume) ──────
#
# On your FIRST session: KAGGLE_DATASET_DIR either doesn't exist or is empty.
# This cell is a no-op and training starts from scratch.
#
# On subsequent sessions: the Kaggle Dataset you created contains the checkpoint
# files saved by the previous session. This cell copies them into PROJECT_DIR
# so train_kaggle.py can auto-resume.
#
# How to save checkpoints to your Kaggle Dataset after each session:
#   Run cell 8b ("Persist checkpoints") BEFORE the session expires.
#   That cell zips checkpoints_kaggle/ and writes them to /kaggle/working/,
#   which becomes a new version of your Kaggle Dataset output.

import os, shutil

CKPT_ARCHIVE = os.path.join(KAGGLE_DATASET_DIR, 'checkpoints_kaggle.tar.gz')
CKPT_DIR_DST = os.path.join(PROJECT_DIR, 'checkpoints_kaggle')
LOG_ARCHIVE  = os.path.join(KAGGLE_DATASET_DIR, 'logs_kaggle.tar.gz')
LOG_DIR_DST  = os.path.join(PROJECT_DIR, 'logs_kaggle')

def _restore_archive(archive_path, dst_dir, label):
    if not os.path.exists(archive_path):
        print(f'  {label}: no archive found at {archive_path} — starting fresh.')
        return False
    os.makedirs(dst_dir, exist_ok=True)
    print(f'  {label}: restoring from {archive_path} ...')
    ret = subprocess.run(
        ['tar', '-xzf', archive_path, '-C', os.path.dirname(dst_dir)],
        capture_output=True, text=True
    )
    if ret.returncode == 0:
        entries = sorted(os.listdir(dst_dir)) if os.path.isdir(dst_dir) else []
        print(f'  {label}: restored — {len(entries)} entries: {entries[-3:] if entries else []}')
        return True
    else:
        print(f'  {label}: restore failed: {ret.stderr}')
        return False

print('Checking persistent storage for previous checkpoints...')
ckpt_restored = _restore_archive(CKPT_ARCHIVE, CKPT_DIR_DST, 'checkpoints_kaggle')
_restore_archive(LOG_ARCHIVE,  LOG_DIR_DST,  'logs_kaggle')

if ckpt_restored:
    latest = sorted(os.listdir(CKPT_DIR_DST))[-1] if os.path.isdir(CKPT_DIR_DST) else None
    print(f'\n✔  Will resume from: {latest}')
    print(f'   (train_kaggle.py will auto-detect the latest checkpoint)')
else:
    print('\n→  No previous checkpoint found — training will start from scratch.')

In [ ]:
# ── 4. Install dependencies ────────────────────────────────────────────────────
!pip install -q tokenizers>=0.19.0 datasets>=2.18.0 huggingface-hub>=0.22.0 psutil
print('Dependencies installed.')

In [ ]:
# ── 5. Set HuggingFace token ───────────────────────────────────────────────────
# Add HF_TOKEN in: Kaggle Account → Settings → Secrets → Add New Secret
import os

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('HF_TOKEN loaded from Kaggle secrets ✔')
    else:
        print('HF_TOKEN secret is empty — add it in Account → Settings → Secrets.')
except Exception:
    # os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'  # ← uncomment + paste as fallback
    print('Could not load Kaggle secrets — continuing without HF_TOKEN (slower downloads).')

In [ ]:
# ── 6. Train tokenizer (runs once; skip if tokenizer_100m/ already exists) ────
# The 150M config reuses the same 32 768-vocab tokenizer as config_100m.
import os, sys, subprocess
sys.path.insert(0, PROJECT_DIR)

tok_path = os.path.join(PROJECT_DIR, 'tokenizer_100m', 'tokenizer.json')
if os.path.exists(tok_path):
    print(f'Tokenizer found — skipping training.')
else:
    print('Training tokenizer (vocab=32768, takes ~20-30 min)...')
    ret = subprocess.run(
        [sys.executable, 'tokenizer_train.py', '--config', 'config_kaggle'],
        cwd=PROJECT_DIR
    )
    if ret.returncode == 0:
        print('Tokenizer saved to tokenizer_100m/')
    else:
        print('⚠  Tokenizer training failed.')

In [ ]:
# ── 7. Show config + resume status ────────────────────────────────────────────
import sys, os
sys.path.insert(0, PROJECT_DIR)
from config_kaggle import model_cfg, train_cfg

d, L, H, Hkv = model_cfg.d_model, model_cfg.n_layers, model_cfg.n_heads, model_cfg.n_kv_heads
B, G, T = train_cfg.batch_size, train_cfg.grad_accum_steps, train_cfg.context_len

print('  ── Model ─────────────────────────────────────')
print(f'  d_model={d}, n_layers={L}, n_heads={H}, kv_heads={Hkv}, d_ff={model_cfg.d_ff}')
print(f'  vocab={model_cfg.vocab_size:,}, context={T}')
print('  ── Training ──────────────────────────────────')
print(f'  Per-GPU batch     : {B} × {G} accum = {B*G} seqs/step')
print(f'  Global eff. toks  : {B*G*T*2:,} tokens/step (2 GPUs)')
print(f'  Max steps         : {train_cfg.max_steps:,}')
print(f'  Session auto-save : {SESSION_HOURS}h ({SESSION_HOURS*60:.0f} min)')
print('  ── Checkpoint status ─────────────────────────')

ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints_kaggle')
if os.path.isdir(ckpt_dir):
    steps = sorted(os.listdir(ckpt_dir))
    print(f'  Latest checkpoint : {steps[-1] if steps else "none"}')
    print(f'  Total checkpoints : {len(steps)}')
else:
    print('  No checkpoints yet — will train from scratch.')

In [ ]:
# ── 8a. Launch DDP training ────────────────────────────────────────────────────
#
# What happens when you click the cell stop button:
#   The kernel sends SIGTERM to the torchrun process group.
#   train_kaggle.py catches it, finishes the current step, saves a checkpoint,
#   and exits cleanly. Then run cell 8b to persist it.
#
# What happens at SESSION_HOURS (8h40m):
#   The training script's timer fires, saves a checkpoint, and exits.
#   The subprocess.run() call below returns, and cell 8b runs automatically.
#
# Checkpoints are also saved every 500 steps (cfg.save_every) during training.

import os, subprocess, sys
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, '-m', 'torch.distributed.run',
    '--nproc_per_node=2',
    '--rdzv_backend=c10d',
    '--rdzv_endpoint=localhost:29500',
    'train_kaggle.py',
    '--session-hours', str(SESSION_HOURS),
]

print('Launching DDP training:')
print(' '.join(cmd))
print('─' * 60)
print(f'Auto-save + exit at {SESSION_HOURS}h ({SESSION_HOURS*60:.0f} min from now).')
print('To stop manually: click the ■ stop button — checkpoint will be saved.')
print('─' * 60)

ret = subprocess.run(cmd, cwd=PROJECT_DIR)

if ret.returncode == 0:
    print('\n✔  Training exited cleanly.')
else:
    print(f'\n⚠  Training exited with code {ret.returncode}.')
print('\n→  Running cell 8b to persist checkpoints...')

In [ ]:
# ── 8b. Persist checkpoints to Kaggle Dataset output ─────────────────────────
#
# /kaggle/working/ is saved as a new version of this notebook's output.
# To make it a DATASET you can mount in future sessions:
#   After this cell completes, go to the notebook's output tab and
#   click "Save Version" → this creates a new dataset version.
#   Future sessions will mount it at KAGGLE_DATASET_DIR.
#
# This cell is intentionally separate from 8a so it runs even if you
# manually stop training early.

import os, subprocess, shutil, time

CKPT_SRC  = os.path.join(PROJECT_DIR, 'checkpoints_kaggle')
LOG_SRC   = os.path.join(PROJECT_DIR, 'logs_kaggle')
SAVE_DIR  = '/kaggle/working'   # this is the notebook output — auto-persisted

def _archive(src_dir, archive_name, label):
    if not os.path.isdir(src_dir) or not os.listdir(src_dir):
        print(f'  {label}: nothing to save (directory empty or missing).')
        return
    dst = os.path.join(SAVE_DIR, archive_name)
    print(f'  {label}: archiving {src_dir} → {dst} ...')
    ret = subprocess.run(
        ['tar', '-czf', dst, '-C', os.path.dirname(src_dir), os.path.basename(src_dir)],
        capture_output=True, text=True
    )
    if ret.returncode == 0:
        size_mb = os.path.getsize(dst) / 1024**2
        print(f'  {label}: saved ({size_mb:.1f} MB)')
    else:
        print(f'  {label}: archive failed: {ret.stderr}')

print('Persisting training state to /kaggle/working/ ...')
_archive(CKPT_SRC, 'checkpoints_kaggle.tar.gz', 'checkpoints_kaggle')
_archive(LOG_SRC,  'logs_kaggle.tar.gz',        'logs_kaggle')

# Show what was saved
saved = [f for f in os.listdir(SAVE_DIR) if f.endswith('.tar.gz')]
print(f'\nFiles in /kaggle/working/:')
for f in sorted(saved):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024**2
    print(f'  {f}  ({size_mb:.1f} MB)')

print()
print('Next steps to resume in a new session:')
print('  1. Click "Save & Run All" or "Save Version" in the top-right.')
print('     This saves the output as a new notebook version.')
print('  2. Go to your Kaggle Dataset (slm-checkpoints) and upload')
print('     the .tar.gz files from this notebook\'s output tab.')
print('  3. Start a new Kaggle session — cell 3 will auto-restore the checkpoint.')

In [ ]:
# ── 9. List checkpoints (run any time to inspect state) ───────────────────────
import os
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints_kaggle')
if os.path.isdir(ckpt_dir):
    entries = sorted(os.listdir(ckpt_dir))
    print(f'Checkpoints in {ckpt_dir}:')
    for e in entries:
        p = os.path.join(ckpt_dir, e)
        if not os.path.isdir(p):
            continue
        files = os.listdir(p)
        sizes = [os.path.getsize(os.path.join(p, f)) / 1024**2 for f in files if f.endswith('.pt')]
        total_mb = sum(sizes)
        import json
        state_file = os.path.join(p, 'train_state.json')
        step_info = ''
        if os.path.exists(state_file):
            with open(state_file) as sf:
                s = json.load(sf)
            step_info = f'  step={s["step"]:,}  tokens={s["tokens_consumed"]/1e6:.1f}M'
        print(f'  {e}  ({total_mb:.0f} MB){step_info}')
else:
    print('No checkpoints found yet.')

In [ ]:
# ── 10. Quick inference check ──────────────────────────────────────────────────
import sys, os, torch
sys.path.insert(0, PROJECT_DIR)

from config_kaggle import model_cfg, train_cfg
from model import SLM
from checkpoint import load_checkpoint
from inference import run_inference_check
from tokenizers import Tokenizer

tok_file = os.path.join(PROJECT_DIR, train_cfg.tokenizer_path, 'tokenizer.json')
if not os.path.exists(tok_file):
    print('Tokenizer not found — run cell 6 first.')
else:
    tokenizer = Tokenizer.from_file(tok_file)
    device    = torch.device('cuda:0')
    model     = SLM(model_cfg).to(device=device, dtype=torch.bfloat16)

    from optimizer import build_optimizers
    muon_opt, adamw_opt = build_optimizers(model, train_cfg)
    step, tokens = load_checkpoint(model, muon_opt, adamw_opt, train_cfg)
    model.eval()
    print(f'Loaded checkpoint at step {step:,}  ({tokens/1e6:.1f}M tokens seen)\n')

    prompts = [
        'The theory of relativity states that',
        'Once upon a time in a kingdom far away',
        'The best way to learn mathematics is',
    ]
    results = run_inference_check(
        model, tokenizer, prompts=prompts,
        max_new_tokens=100, temperature=0.8, top_k=40,
    )
    for prompt, gen in zip(prompts, results):
        print(f'PROMPT : {prompt}')
        print(f'OUTPUT : {gen}\n')

In [ ]:
# ── 11. Plot training loss from JSONL log ─────────────────────────────────────
import json, os
import matplotlib.pyplot as plt

log_dir = os.path.join(PROJECT_DIR, 'logs_kaggle')
records = []
if os.path.isdir(log_dir):
    for fname in sorted(os.listdir(log_dir)):
        if fname.endswith('.jsonl'):
            with open(os.path.join(log_dir, fname)) as f:
                for line in f:
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass

if not records:
    print('No log records found yet.')
else:
    steps      = [r['step']     for r in records if 'loss' in r]
    train_loss = [r['loss']     for r in records if 'loss' in r]
    val_steps  = [r['step']     for r in records if r.get('val_loss') is not None]
    val_loss   = [r['val_loss'] for r in records if r.get('val_loss') is not None]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(steps, train_loss, lw=0.8, alpha=0.7, label='Train loss')
    if val_steps:
        ax.plot(val_steps, val_loss, lw=1.5, marker='o', ms=3, label='Val loss')
    ax.set_xlabel('Step')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title('SLM 150M — Kaggle 2× T4 DDP Training')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'training_loss.png'), dpi=150)
    plt.show()
    print(f'Last train loss : {train_loss[-1]:.4f} at step {steps[-1]:,}')
    if val_steps:
        print(f'Last val loss   : {val_loss[-1]:.4f} at step {val_steps[-1]:,}')

---

## Session resume guide

### Resuming in a new Kaggle session

```
Session N ends (timer or manual stop)
  └─ train_kaggle.py saves checkpoint to checkpoints_kaggle/
  └─ Cell 8b archives checkpoints_kaggle/ → /kaggle/working/checkpoints_kaggle.tar.gz
  └─ You upload the .tar.gz to your 'slm-checkpoints' Kaggle Dataset

Session N+1 starts
  └─ Cell 3 restores from /kaggle/input/slm-checkpoints/checkpoints_kaggle.tar.gz
  └─ Cell 8a launches torchrun — train_kaggle.py auto-resumes from latest checkpoint
```

### Shutdown behaviour

| Trigger | Signal | What happens |
|---|---|---|
| ■ Stop button | `SIGTERM` | Caught by `_ShutdownHandler` → finish current step → save checkpoint → exit |
| Kernel interrupt (Interrupt button) | `SIGINT` | Same as above |
| 8h40m wall-clock | timer | `_SessionTimer.expired` fires → same save-and-exit path |
| Kaggle hard kill (12h) | `SIGKILL` | **Cannot be caught** — rely on the periodic 500-step checkpoint |

> `SIGKILL` cannot be intercepted by any process. The 500-step periodic checkpoint (`save_every=500`)
> means you lose at most 500 steps of work if the session hard-kills.
> At ~80k tokens/step that is ~65M tokens — roughly 1–2% of the total training budget.

### Tips

| Issue | Fix |
|---|---|
| Only 1 GPU | Settings → Accelerator → **GPU T4 × 2** |
| `NCCL error` on start | Re-run cell 8a; port 29500 busy → change `rdzv_endpoint` to `:29501` |
| CUDA OOM | Reduce `batch_size` to 4 in `config_kaggle.py` |
| `ModuleNotFoundError` | `os.chdir(PROJECT_DIR)` then re-run the cell |
| Slow HF downloads | Add `HF_TOKEN` in Account → Settings → Secrets |
| Want longer sessions | Change `SESSION_HOURS` in cell 0 (max ~11h before Kaggle kills) |
| Want more frequent saves | Lower `save_every` in `config_kaggle.py` (default 500 steps) |